# Required Task 15 — Fine-Tuning for Credit-Card Complaint Classification & Resolution

**Objective:** Fine-tune transformer models on the `credit_card_qa` dataset to:
1. **Model 1 — Policy Category Prediction:** Given a complaint, predict the `policy_category`.
2. **Model 2 — Resolution Generation:** Given a complaint, generate the `resolution`.

We compare **DistilBERT-base-uncased** (encoder, ideal for classification) with **DistilGPT-2** (decoder, ideal for generation) on both tasks.

| Task | DistilBERT Approach | DistilGPT-2 Approach |
|---|---|---|
| Policy Category | Sequence classification head | Conditional text generation |
| Resolution | Sequence classification head (over resolution templates) | Conditional text generation |

**Protocol:** 60 records for training → evaluate on all 80 records.

**Dataset:** [huggingface.co/datasets/priyaannamani/credit_card_qa](https://huggingface.co/datasets/priyaannamani/credit_card_qa)

## 1 — Environment Setup

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn rouge-score nltk pandas

In [ ]:
import warnings, os, random, gc, shutil, glob
import numpy as np, pandas as pd, torch
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Disk helper: nuke checkpoints + trainer artefacts, free GPU ──
def cleanup(*objs, dirs=None):
    """Delete python objects, purge checkpoint dirs, free GPU."""
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    for d in (dirs or []):
        if os.path.isdir(d):
            shutil.rmtree(d, ignore_errors=True)
    # Report disk
    st = os.statvfs('/')
    used_gb = (st.f_blocks - st.f_bfree) * st.f_frsize / 1e9
    total_gb = st.f_blocks * st.f_frsize / 1e9
    print(f"  Disk: {used_gb:.1f} GB used / {total_gb:.1f} GB total")

cleanup()  # baseline disk report

## 2 — Load & Explore the Dataset

In [ ]:
try:
    from datasets import load_dataset
    hf_ds = load_dataset('priyaannamani/credit_card_qa')
    df = hf_ds['train'].to_pandas()
    print("Loaded from HuggingFace Hub.")
except Exception:
    df = pd.read_csv('credit_card_qa.csv')
    print("Loaded from local CSV.")

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
print(f"Unique policy_category values: {df['policy_category'].nunique()}")
print(f"Unique resolution values:      {df['resolution'].nunique()}")
print(f"\nComplaint length  — mean: {df['complaint'].str.len().mean():.0f} chars")
print(f"Resolution length — mean: {df['resolution'].str.len().mean():.0f} chars")
print("\nPolicy category distribution:")
print(df['policy_category'].value_counts())

## 3 — Train / Full Split

60 records for fine-tuning; evaluate on **all 80** records.

In [ ]:
# Label encodings for policy_category
categories = sorted(df['policy_category'].unique())
cat2id = {c: i for i, c in enumerate(categories)}
id2cat = {i: c for c, i in cat2id.items()}
df['cat_label'] = df['policy_category'].map(cat2id)
NUM_CATEGORIES = len(categories)
print(f"Number of policy categories: {NUM_CATEGORIES}")

train_df   = df.sample(n=60, random_state=SEED)
eval_df    = df.copy()                       # all 80
holdout_df = df.drop(train_df.index)          # 20 unseen
print(f"Train: {len(train_df)} | Eval (all): {len(eval_df)} | Held-out: {len(holdout_df)}")

---
# TASK A — Predict `policy_category` from `complaint`
---
## 4A — DistilBERT for Policy-Category Classification

In [ ]:
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    Trainer, TrainingArguments
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score

bert_tok = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_for_bert(examples):
    return bert_tok(examples['complaint'], truncation=True,
                    padding='max_length', max_length=192)

train_ds_bert = Dataset.from_pandas(
    train_df[['complaint','cat_label']].rename(columns={'cat_label':'label'}))
eval_ds_bert = Dataset.from_pandas(
    eval_df[['complaint','cat_label']].rename(columns={'cat_label':'label'}))

train_ds_bert = train_ds_bert.map(tokenize_for_bert, batched=True)
eval_ds_bert  = eval_ds_bert.map(tokenize_for_bert, batched=True)
train_ds_bert.set_format('torch', columns=['input_ids','attention_mask','label'])
eval_ds_bert.set_format('torch', columns=['input_ids','attention_mask','label'])
print(f"Tokenised — train: {len(train_ds_bert)}, eval: {len(eval_ds_bert)}")

In [ ]:
bert_cat_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=NUM_CATEGORIES)
bert_cat_model.to(DEVICE)

def compute_metrics_cls(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1_weighted': f1_score(labels, preds, average='weighted', zero_division=0)}

args_bert_cat = TrainingArguments(
    output_dir='./tmp_bert_cat',
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',           # ★ ZERO checkpoints — saves ~5 GB
    logging_steps=20,
    seed=SEED, report_to='none',
)

trainer_bert_cat = Trainer(
    model=bert_cat_model, args=args_bert_cat,
    train_dataset=train_ds_bert, eval_dataset=eval_ds_bert,
    compute_metrics=compute_metrics_cls)

print("Training DistilBERT for policy_category...")
trainer_bert_cat.train()
print("Done.")

In [ ]:
# ── Evaluate on ALL 80 records ──
bert_cat_results = trainer_bert_cat.evaluate()
print("=" * 52)
print("DistilBERT — Policy Category — All 80 Records")
print("=" * 52)
print(f"Accuracy:    {bert_cat_results['eval_accuracy']:.4f}")
print(f"F1 Weighted: {bert_cat_results['eval_f1_weighted']:.4f}")

preds_out = trainer_bert_cat.predict(eval_ds_bert)
bert_cat_pred_labels = np.argmax(preds_out.predictions, axis=-1)
bert_cat_pred_names  = [id2cat[p] for p in bert_cat_pred_labels]

print("\nClassification Report:")
print(classification_report(eval_df['policy_category'].tolist(),
                            bert_cat_pred_names, zero_division=0))

In [ ]:
# ── Held-out 20 unseen ──
ho_ds = Dataset.from_pandas(
    holdout_df[['complaint','cat_label']].rename(columns={'cat_label':'label'}))
ho_ds = ho_ds.map(tokenize_for_bert, batched=True)
ho_ds.set_format('torch', columns=['input_ids','attention_mask','label'])
ho_res = trainer_bert_cat.evaluate(ho_ds)
print(f"Held-out (20 unseen) — DistilBERT Policy Category:")
print(f"  Accuracy: {ho_res['eval_accuracy']:.4f}  F1: {ho_res['eval_f1_weighted']:.4f}")

In [ ]:
# ★ Free everything before next model
cleanup(bert_cat_model, trainer_bert_cat, dirs=['./tmp_bert_cat'])

## 4B — DistilGPT-2 for Policy-Category (Generative Approach)

In [ ]:
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

gpt2_tok = GPT2TokenizerFast.from_pretrained('distilgpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token

SEP_CAT = ' -> Category: '
MAX_LEN_CAT = 256

def build_gpt2_cat_texts(dataframe):
    return [f"Complaint: {r['complaint']}{SEP_CAT}{r['policy_category']}{gpt2_tok.eos_token}"
            for _, r in dataframe.iterrows()]

def tokenize_gpt2_cat(examples):
    out = gpt2_tok(examples['text'], truncation=True,
                   padding='max_length', max_length=MAX_LEN_CAT)
    out['labels'] = out['input_ids'].copy()
    return out

train_ds_gpt2_cat = Dataset.from_dict({'text': build_gpt2_cat_texts(train_df)})
train_ds_gpt2_cat = train_ds_gpt2_cat.map(tokenize_gpt2_cat, batched=True)
train_ds_gpt2_cat.set_format('torch', columns=['input_ids','attention_mask','labels'])
print(f"GPT-2 category training set: {len(train_ds_gpt2_cat)} samples")

In [ ]:
gpt2_cat_model = GPT2LMHeadModel.from_pretrained('distilgpt2')
gpt2_cat_model.config.pad_token_id = gpt2_tok.eos_token_id
gpt2_cat_model.to(DEVICE)

args_gpt2_cat = TrainingArguments(
    output_dir='./tmp_gpt2_cat',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy='no',           # ★ NO checkpoints
    seed=SEED, report_to='none',
)

trainer_gpt2_cat = Trainer(
    model=gpt2_cat_model, args=args_gpt2_cat,
    train_dataset=train_ds_gpt2_cat)

print("Training DistilGPT-2 for policy_category...")
trainer_gpt2_cat.train()
print("Done.")

In [ ]:
def predict_category_gpt2(complaint_text, model, tokenizer, max_new=25):
    prompt = f"Complaint: {complaint_text}{SEP_CAT}"
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=220).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                             pad_token_id=tokenizer.eos_token_id,
                             do_sample=False)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if SEP_CAT.strip() in decoded:
        pred = decoded.split(SEP_CAT.strip())[-1].strip().split('\n')[0]
        for stop in ['Complaint:', ' ->']:
            if stop in pred: pred = pred.split(stop)[0].strip()
        return pred
    return decoded.strip()

# Quick test
s = eval_df.iloc[0]
print(f"GT:   {s['policy_category']}")
print(f"Pred: {predict_category_gpt2(s['complaint'], gpt2_cat_model, gpt2_tok)}")

In [ ]:
# Evaluate on all 80
gpt2_cat_preds = [predict_category_gpt2(r['complaint'], gpt2_cat_model, gpt2_tok)
                  for _, r in eval_df.iterrows()]

gpt2_cat_exact = [p.strip().lower() == g.strip().lower()
                  for p, g in zip(gpt2_cat_preds, eval_df['policy_category'])]
gpt2_cat_fuzzy = [g.strip().lower() in p.strip().lower()
                  or p.strip().lower() in g.strip().lower()
                  for p, g in zip(gpt2_cat_preds, eval_df['policy_category'])]
gpt2_cat_accuracy   = sum(gpt2_cat_exact) / len(gpt2_cat_exact)
gpt2_cat_fuzzy_acc  = sum(gpt2_cat_fuzzy) / len(gpt2_cat_fuzzy)

print("=" * 52)
print("DistilGPT-2 — Policy Category — All 80 Records")
print("=" * 52)
print(f"Exact Match Accuracy: {gpt2_cat_accuracy:.4f}")
print(f"Fuzzy Match Accuracy: {gpt2_cat_fuzzy_acc:.4f}")

print("\nSample Predictions:")
for i in range(5):
    gt = eval_df.iloc[i]['policy_category']
    pr = gpt2_cat_preds[i]
    m = '✓' if gpt2_cat_exact[i] else ('~' if gpt2_cat_fuzzy[i] else '✗')
    print(f"  [{m}] GT: {gt:40s} | Pred: {pr}")

In [ ]:
# Held-out
ho_gpt2_cat = [predict_category_gpt2(r['complaint'], gpt2_cat_model, gpt2_tok)
               for _, r in holdout_df.iterrows()]
ho_exact = [p.strip().lower() == g.strip().lower()
            for p, g in zip(ho_gpt2_cat, holdout_df['policy_category'])]
print(f"Held-out (20) — DistilGPT-2 Exact Match: {sum(ho_exact)/len(ho_exact):.4f}")

In [ ]:
# ★ Free everything
cleanup(gpt2_cat_model, trainer_gpt2_cat, dirs=['./tmp_gpt2_cat'])

---
# TASK B — Predict `resolution` from `complaint`

Resolutions are free-text (78 unique / 80 records) → fundamentally a **generation** task.

---
## 5A — DistilGPT-2 for Resolution Generation

In [ ]:
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

gpt2_tok = GPT2TokenizerFast.from_pretrained('distilgpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token

SEP_RES = ' -> Resolution: '
MAX_LEN_RES = 350

def build_gpt2_res_texts(dataframe):
    return [f"Complaint: {r['complaint']}{SEP_RES}{r['resolution']}{gpt2_tok.eos_token}"
            for _, r in dataframe.iterrows()]

def tokenize_gpt2_res(examples):
    out = gpt2_tok(examples['text'], truncation=True,
                   padding='max_length', max_length=MAX_LEN_RES)
    out['labels'] = out['input_ids'].copy()
    return out

train_ds_gpt2_res = Dataset.from_dict({'text': build_gpt2_res_texts(train_df)})
train_ds_gpt2_res = train_ds_gpt2_res.map(tokenize_gpt2_res, batched=True)
train_ds_gpt2_res.set_format('torch', columns=['input_ids','attention_mask','labels'])
print(f"GPT-2 resolution training set: {len(train_ds_gpt2_res)} samples")

In [ ]:
gpt2_res_model = GPT2LMHeadModel.from_pretrained('distilgpt2')
gpt2_res_model.config.pad_token_id = gpt2_tok.eos_token_id
gpt2_res_model.to(DEVICE)

args_gpt2_res = TrainingArguments(
    output_dir='./tmp_gpt2_res',
    num_train_epochs=20,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=20,
    save_strategy='no',           # ★ NO checkpoints
    seed=SEED, report_to='none',
)

trainer_gpt2_res = Trainer(
    model=gpt2_res_model, args=args_gpt2_res,
    train_dataset=train_ds_gpt2_res)

print("Training DistilGPT-2 for resolution generation...")
trainer_gpt2_res.train()
print("Done.")

In [ ]:
def predict_resolution_gpt2(complaint_text, model, tokenizer, max_new=70):
    prompt = f"Complaint: {complaint_text}{SEP_RES}"
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=260).to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new,
                             pad_token_id=tokenizer.eos_token_id,
                             do_sample=False)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if SEP_RES.strip() in decoded:
        pred = decoded.split(SEP_RES.strip())[-1].strip().split('\n')[0]
        for stop in ['Complaint:', ' ->']:
            if stop in pred: pred = pred.split(stop)[0].strip()
        return pred
    return decoded.strip()

s = eval_df.iloc[0]
print(f"GT:   {s['resolution']}")
print(f"Pred: {predict_resolution_gpt2(s['complaint'], gpt2_res_model, gpt2_tok)}")

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

gpt2_res_preds = []
gpt2_rouge = {'rouge1':[], 'rouge2':[], 'rougeL':[]}

for _, row in eval_df.iterrows():
    pred = predict_resolution_gpt2(row['complaint'], gpt2_res_model, gpt2_tok)
    gpt2_res_preds.append(pred)
    sc = scorer.score(row['resolution'], pred)
    for k in gpt2_rouge: gpt2_rouge[k].append(sc[k].fmeasure)

print("=" * 55)
print("DistilGPT-2 — Resolution — All 80 Records")
print("=" * 55)
for k in gpt2_rouge:
    print(f"  {k}: {np.mean(gpt2_rouge[k]):.4f}")

print("\nSample Predictions:")
for i in range(5):
    print(f"\n  [{i+1}] GT:   {eval_df.iloc[i]['resolution'][:100]}")
    print(f"      Pred: {gpt2_res_preds[i][:100]}")

In [ ]:
# Held-out ROUGE
ho_rouge = {'rouge1':[], 'rouge2':[], 'rougeL':[]}
for _, row in holdout_df.iterrows():
    pred = predict_resolution_gpt2(row['complaint'], gpt2_res_model, gpt2_tok)
    sc = scorer.score(row['resolution'], pred)
    for k in ho_rouge: ho_rouge[k].append(sc[k].fmeasure)
print("Held-out (20) — DistilGPT-2 Resolution ROUGE:")
for k in ho_rouge: print(f"  {k}: {np.mean(ho_rouge[k]):.4f}")

In [ ]:
cleanup(gpt2_res_model, trainer_gpt2_res, dirs=['./tmp_gpt2_res'])

## 5B — DistilBERT for Resolution (Classification Over Unique Resolutions)

78 unique resolutions / 80 records → most classes have exactly 1 example. This deliberately demonstrates why framing generation as classification fails here.

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

bert_tok = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_for_bert(examples):
    return bert_tok(examples['complaint'], truncation=True,
                    padding='max_length', max_length=192)

resolutions = sorted(df['resolution'].unique())
res2id = {r: i for i, r in enumerate(resolutions)}
id2res = {i: r for r, i in res2id.items()}
NUM_RES = len(resolutions)
print(f"Unique resolution classes: {NUM_RES}")

train_ds_bert_res = Dataset.from_pandas(
    train_df[['complaint']].assign(label=train_df['resolution'].map(res2id)))
eval_ds_bert_res = Dataset.from_pandas(
    eval_df[['complaint']].assign(label=eval_df['resolution'].map(res2id)))

train_ds_bert_res = train_ds_bert_res.map(tokenize_for_bert, batched=True)
eval_ds_bert_res  = eval_ds_bert_res.map(tokenize_for_bert, batched=True)
train_ds_bert_res.set_format('torch', columns=['input_ids','attention_mask','label'])
eval_ds_bert_res.set_format('torch', columns=['input_ids','attention_mask','label'])
print(f"Datasets — train: {len(train_ds_bert_res)}, eval: {len(eval_ds_bert_res)}")

In [ ]:
bert_res_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=NUM_RES)
bert_res_model.to(DEVICE)

args_bert_res = TrainingArguments(
    output_dir='./tmp_bert_res',
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',           # ★ NO checkpoints
    logging_steps=20,
    seed=SEED, report_to='none',
)

trainer_bert_res = Trainer(
    model=bert_res_model, args=args_bert_res,
    train_dataset=train_ds_bert_res, eval_dataset=eval_ds_bert_res,
    compute_metrics=compute_metrics_cls)

print("Training DistilBERT for resolution classification...")
trainer_bert_res.train()
print("Done.")

In [ ]:
bert_res_results = trainer_bert_res.evaluate()
preds_out = trainer_bert_res.predict(eval_ds_bert_res)
bert_res_pred_labels = np.argmax(preds_out.predictions, axis=-1)
bert_res_pred_texts  = [id2res[p] for p in bert_res_pred_labels]

# ROUGE for fair comparison
bert_rouge = {'rouge1':[], 'rouge2':[], 'rougeL':[]}
for pred, gt in zip(bert_res_pred_texts, eval_df['resolution']):
    sc = scorer.score(gt, pred)
    for k in bert_rouge: bert_rouge[k].append(sc[k].fmeasure)

print("=" * 55)
print("DistilBERT — Resolution (as Classification) — All 80")
print("=" * 55)
print(f"Accuracy:    {bert_res_results['eval_accuracy']:.4f}")
print(f"F1 Weighted: {bert_res_results['eval_f1_weighted']:.4f}")
print("ROUGE (for comparison with GPT-2):")
for k in bert_rouge:
    print(f"  {k}: {np.mean(bert_rouge[k]):.4f}")

print("\nSample Predictions:")
for i in range(5):
    print(f"\n  [{i+1}] GT:   {eval_df.iloc[i]['resolution'][:100]}")
    print(f"      Pred: {bert_res_pred_texts[i][:100]}")

In [ ]:
cleanup(bert_res_model, trainer_bert_res, dirs=['./tmp_bert_res'])

---
# 6 — Comparative Analysis
---

In [ ]:
# ═══════════════════════════════════════════════════
#  TASK A: POLICY CATEGORY
# ═══════════════════════════════════════════════════
print("═" * 65)
print(" TASK A: POLICY CATEGORY PREDICTION — COMPARATIVE RESULTS")
print("═" * 65)
print(f"{'Metric':<25} {'DistilBERT':>15} {'DistilGPT-2':>15}")
print("─" * 65)
print(f"{'Accuracy':<25} {bert_cat_results['eval_accuracy']:>15.4f} {gpt2_cat_accuracy:>15.4f}")
print(f"{'F1 Weighted':<25} {bert_cat_results['eval_f1_weighted']:>15.4f} {'N/A':>15}")
print(f"{'Fuzzy Match':<25} {'N/A':>15} {gpt2_cat_fuzzy_acc:>15.4f}")
print(f"{'Approach':<25} {'Classification':>15} {'Generation':>15}")
print(f"{'# Parameters':<25} {'~66M':>15} {'~82M':>15}")
print()

# ═══════════════════════════════════════════════════
#  TASK B: RESOLUTION
# ═══════════════════════════════════════════════════
print("═" * 65)
print(" TASK B: RESOLUTION PREDICTION — COMPARATIVE RESULTS")
print("═" * 65)
print(f"{'Metric':<25} {'DistilBERT':>15} {'DistilGPT-2':>15}")
print("─" * 65)
print(f"{'Accuracy (exact)':<25} {bert_res_results['eval_accuracy']:>15.4f} {'N/A':>15}")
print(f"{'F1 Weighted':<25} {bert_res_results['eval_f1_weighted']:>15.4f} {'N/A':>15}")
for k in ['rouge1','rouge2','rougeL']:
    b = np.mean(bert_rouge[k]); g = np.mean(gpt2_rouge[k])
    print(f"{k:<25} {b:>15.4f} {g:>15.4f}")
print(f"{'Approach':<25} {'Classification':>15} {'Generation':>15}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Task A bar chart ──
ax = axes[0]
models = ['DistilBERT\n(Classification)', 'DistilGPT-2\n(Generation)']
accs = [bert_cat_results['eval_accuracy'], gpt2_cat_accuracy]
cols = ['#4C72B0', '#DD8452']
bars = ax.bar(models, accs, color=cols, width=0.5, edgecolor='white')
for b, v in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{v:.2%}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.15); ax.set_ylabel('Accuracy')
ax.set_title('Task A: Policy Category — Accuracy', fontweight='bold')
ax.spines[['top','right']].set_visible(False)

# ── Task B bar chart ──
ax = axes[1]
rk = ['rouge1','rouge2','rougeL']
br = [np.mean(bert_rouge[k]) for k in rk]
gr = [np.mean(gpt2_rouge[k]) for k in rk]
x = np.arange(len(rk)); w = 0.3
b1 = ax.bar(x-w/2, br, w, label='DistilBERT', color='#4C72B0')
b2 = ax.bar(x+w/2, gr, w, label='DistilGPT-2', color='#DD8452')
for bar, val in zip(list(b1)+list(b2), br+gr):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['ROUGE-1','ROUGE-2','ROUGE-L'])
ax.set_ylim(0, 1.15); ax.set_ylabel('F-measure')
ax.set_title('Task B: Resolution — ROUGE Scores', fontweight='bold')
ax.legend(); ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fine_tuning_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
# 7 — Narrative: Insights from Fine-Tuning

---

### Key Findings

**1. Architecture Matters — Match the Model to the Task**

DistilBERT (encoder-only) is architecturally suited for classification: it produces a single embedding of the entire input, which a classification head maps to discrete labels. For the policy-category task — with 21 well-defined classes — DistilBERT is expected to outperform a generative model that must reconstruct the exact label string character by character.

DistilGPT-2 (decoder-only) is designed for sequential text generation, making it the natural choice for the resolution task where the output is free-form text that varies across almost every record.

**2. Small Data, High Memorisation**

With only 60 training examples both models face severe data scarcity. Training loss drops quickly — indicating the models memorise the training set within a few epochs. Performance on the 60 seen examples is high, but generalisation to the 20 unseen records is the real test. Any gap between the all-80 and held-out-20 evaluations reveals how much the models rely on memorisation versus transferable patterns.

**3. Classification vs. Generation for Categories**

DistilBERT's classification head constrains output to valid category labels, guaranteeing well-formed predictions. DistilGPT-2 can generate any text, including misspelled or hallucinated categories. The fuzzy-match metric partially compensates, but in production a classification approach is more reliable for closed-set predictions.

**4. Resolution is Fundamentally a Generation Problem**

With 78 unique resolutions across 80 records, framing this as classification gives DistilBERT a 78-class problem where most classes have exactly one training example — an almost impossible classification task. DistilGPT-2 can learn the *style and vocabulary* of resolutions and compose novel-but-plausible text even if it does not exactly match the reference. ROUGE scores (measuring n-gram overlap rather than exact match) are the appropriate metric here.

**5. Practical Takeaways**

- For **fixed-label prediction** (policy category), prefer an encoder model with a classification head.
- For **open-ended generation** (resolution text), prefer a decoder model fine-tuned with a causal-LM objective.
- With tiny datasets (< 100 examples), fine-tuning is essentially few-shot memorisation. For production, consider data augmentation, RAG, or larger pre-trained models with in-context learning.
- No single architecture dominates all tasks — the right choice depends on whether the output space is closed (classification) or open (generation).